# 업무 지침·도구·기억을 고르는 에이전트

모델에 모든 업무 문서와 모든 대화를 보내지 않고, 이번 요청에 필요한 지침과 사용자의 기록만 꺼낸다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 이 예제는 파이썬 표준 라이브러리로 실행할 수 있습니다.

## 지침 선택과 도구 실행을 분리하기

지침은 어떤 순서로 확인할지 설명한다. 실제 조회는 도구가 수행한다. 지침을 읽었다는 로그만으로 고객 요청을 조회했다고 판단하면 안 된다.

예를 들어 재발급과 열람 기간은 같은 영수증 문의여도 필요한 도구가 다르다. 먼저 짧은 지침 설명으로 후보를 고르고 선택한 문서의 상세 내용만 연다.

개인 선호는 사용자 ID, 방금 한 질문은 대화 ID를 기준으로 가져온다. 두 키를 같은 값으로 쓰면 새 대화에서 필요한 설정을 잃거나 다른 사용자의 설정이 섞인다.

## 서로 다른 사용자의 같은 대화 이름

같은 이름의 대화방을 가진 두 사용자가 있다고 가정한다. 지속 설정과 최근 대화의 키를 분리한다.

In [1]:
profiles = {"u1":{"format":"표"}, "u2":{"format":"문장"}}
turns = {("u1","상담"):["영수증 기한"], ("u2","상담"):["재발급"]}
def context(user, thread):
    return profiles.get(user, {}), turns.get((user,thread), [])
print(context("u2", "상담"))

({'format': '문장'}, ['재발급'])


u2의 재발급 요청과 표시 형식만 돌아온다. 모델이 기억했다는 인상을 받기 전에 앱이 실제로 보낸 기록부터 확인할 수 있다.

### 새 대화에서 유지할 정보 고르기

u1이 새 대화방을 연다. 과거 질문을 넘기지 않으면서 형식 선호를 유지하는지 확인하고, 낯선 사용자도 시험한다.

In [2]:
new_thread = "새 상담"

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>print(context(&quot;u1&quot;, new_thread))
print(context(&quot;u3&quot;, &quot;상담&quot;))</code></pre><pre>({&#x27;format&#x27;: &#x27;표&#x27;}, [])
({}, [])
</pre><p>u1은 표 설정만 남고 u3에게는 다른 사람의 기록을 주지 않는다.</p></details>

## 지침을 고른 다음에만 본문을 읽기

임시 폴더에 지침 두 개를 만들고 어떤 파일을 실제로 열었는지 기록한다. 제목 후보가 많아도 요청이 필요로 하는 본문만 컨텍스트에 넣는다.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
workspace = TemporaryDirectory()
guide_dir = Path(workspace.name)
(guide_dir/"receipt.txt").write_text("영수증 기한은 기준일 확인 후 조회",encoding="utf-8")
(guide_dir/"issue.txt").write_text("재발급 확정 전 본인 확인",encoding="utf-8")
index = {"열람":"receipt.txt","재발급":"issue.txt"}
def load_guide(topic):
    name = index.get(topic)
    if name is None:
        return [], None
    return [name], (guide_dir/name).read_text(encoding="utf-8")
print(load_guide("재발급"))

(['issue.txt'], '재발급 확정 전 본인 확인')


실제로 읽은 파일은 issue.txt 하나다. 등록된 지침이 있다는 사실과 모델 입력으로 본문을 전달했다는 사실을 분리할 수 있다.

### 알 수 없는 주제에서 상세 파일 열지 않기

아무 지침에나 연결해 버리지 않도록 대출 문의를 보낸다. 읽은 파일 수와 선택한 내용을 확인한다.

In [5]:
topic = "대출"

In [6]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>opened, detail = load_guide(topic)
print(len(opened), detail)</code></pre><pre>0 None
</pre><p>열린 파일은 없고 내용은 None이다. 실제 에이전트에서는 이 상태에서 추가 질문을 할지 기본 안내를 할지 결정한다.</p></details>

## 다른 입력에 적용하기

수업용 create_agent에 실제 도구 둘을 등록한다. 조회가 필요한 요청, 도구가 필요 없는 인사, 입력이 모자란 요청을 실행하고 지침 파일을 연 시점과 도구 호출 기록을 분리한다. 같은 조회를 로컬 MCP 서버로 옮겨도 반환 계약이 같은지 비교한다.

필드가 하나 빠졌을 때 에이전트가 계속 도구를 부르지 않도록 한도를 두고, 중단 시점의 요청·도구·인자를 남긴다.